In [1]:
import sys

print(sys.executable)

/Users/Work/VSCodeProjects/.venv/bin/python


In [6]:
import pandas as pd
from scipy import stats
import math
import numpy as np

# Task 1

According to open source data, the average salary in the country in 2022 was 64 200 руб. You have a sample for 2023. You predict that the average salary will decrease by 700. Test your hypothesis against the left-handed alternative with a significance level of 0.05

### First, let's inspect the data.

We assume that each employee appears only once in the dataset and that the sample was collected independently at random.

In [ ]:

df = pd.read_csv("Salary.csv")
df = df.dropna()
print("\n----- Describe salary data: \n")
df["Зарплата"].describe()


----- Describe salary data: 



count      8122.000000
mean      62996.158828
std       25690.784074
min       12038.000000
25%       43301.000000
50%       61778.000000
75%       80983.250000
max      155021.000000
Name: Зарплата, dtype: float64

Let's inspect potential outliers.

In [56]:
q1 = df["Зарплата"].quantile(0.25)  # 25th percentile (0.25 quantile)
q3 = df["Зарплата"].quantile(0.75)  # 75th percentile (0.75 quantile)

iqr = q3 - q1                       # interquartile range
lower = q1 - 1.5*iqr                # Whiskers: extend to the most extreme points
upper = q3 + 1.5*iqr

outliers = df[(df["Зарплата"] < lower) | (df["Зарплата"] > upper)]

print("\n----- Describe outliers:")
print("\nLower part count:", outliers[outliers["Зарплата"] <= q1]["Зарплата"].count())
print("\nUpper part count:", outliers[outliers["Зарплата"] >= q3]["Зарплата"].count())



----- Describe outliers:

Lower part count: 0

Upper part count: 28


The outliers appear to be a natural part of the population. Salary distributions are naturally right-skewed. There are only 28 potential outliers among 8 122 observations (approximately 0.34% of the sample). We keep these observations because they are likely to represent genuine salaries rather than data errors.

The sample size is large (n = 8122). Therefore, by the Central Limit Theorem, the sampling distribution of the sample mean is approximately normal even if the population distribution is not.

### Seecondly, let's state the null and alternative hypotheses. 

Null hypothesis $H_0: \mu_0 = 63500$

Alternative hypothesis $H_1​: μ_0 < 63500$

### Choosing the test

We want to test the population mean from one sample. Since the population variance is unknown, a one-sample t-test is appropriate.

The test statistic is 
$$
T = \frac{\bar X - \mu_0}{S/\sqrt{n}}
$$
 which follows Student's t-distribution with n−1 degrees of freedom under the null hypothesis.

In [66]:
salary = df["Зарплата"]

mu = salary.mean()
std = salary.std()
n = len(salary)
print(f"The data has mean = {round(mu, 2)}, std = {round(std, 2)}, n = {n}")

The data has mean = 62996.16, std = 25690.78, n = 8122


In [69]:
mu_0 = 64_200 - 700
alpha = 0.05
critical_value = stats.t.ppf(alpha, df=n-1)
print(f"Critical value is {round(critical_value, 2)}")

Critical value is -1.65


In [70]:
t = (mu - mu_0) / (std / math.sqrt(n))
print(f"The value of the test statistics is {round(t, 2)}")

The value of the test statistics is -1.77


Since the test statistic (-1.77) is less than the critical value (-1.645), we reject the null hypothesis at the 5% significance level. There is sufficient evidence that the mean salary in 2023 is lower than 63,500 rubles.

Let's compute the p-value

In [67]:
p_value = stats.t.cdf(t, df=n-1)
print(f"The p-value is {round(p_value, 4)} -- the smallest significance level at which the null hypothesis would be rejected.")

The p-value is 0.0386 -- the smallest significance level at which the null hypothesis would be rejected.


### Fast-track workflow (Equivalent implementation using SciPy).

In [71]:
salary = df["Зарплата"].dropna()

mu0 = 63_500

result = stats.ttest_1samp(
    salary,
    popmean=mu0,
    alternative="less"
)

print(f"t statistic = {result.statistic:.3f}")
print(f"p-value = {result.pvalue:.4f}")

if result.pvalue <= alpha:
    print(f"Since p-value <= {alpha}, reject H0.")
else:
    print(f"Since p-value > {alpha}, fail to reject H0.")

t statistic = -1.767
p-value = 0.0386
Since p-value <= 0.05, reject H0.
